In [70]:
import pandas as pd
import numpy as np

In [71]:
csv_file_path ="../data/dif_ohlcv_1m.csv"
df = pd.read_csv(csv_file_path)


In [72]:
# 1) standardize timestamp column name and type (DBN uses UTC ns)
ts_col = 'ts' if 'ts' in df.columns else 'ts_event'
df = df.rename(columns={ts_col: 'ts'}).copy()
df['ts'] = pd.to_datetime(df['ts'], utc=True)

# 2) set index and temporarily view in New York time
df_idx = df.set_index('ts')
df_ny = df_idx.tz_convert('America/New_York')

# 3) keep only RTH (handles DST automatically)
rth_ny = df_ny.between_time('09:30', '16:00')

# 4) convert back to UTC for storage/consistency (optional)
rth = rth_ny.tz_convert('UTC').reset_index()   # 'ts' is back in UTC


In [73]:
rth.head()

,ts,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
0,2024-10-15 13:30:00+00:00,33,2,10047,85.39,85.87,85.370,85.820,122569,LRCX
1,2024-10-15 13:30:00+00:00,33,2,753,213.52,214.41,213.520,214.270,83683,AMAT
2,2024-10-15 13:30:00+00:00,33,2,16727,281.16,281.83,280.700,281.040,8708,V
3,2024-10-15 13:30:00+00:00,33,2,10193,505.64,506.99,505.565,506.170,2367,MA
4,2024-10-15 13:31:00+00:00,33,2,10193,506.77,506.88,506.170,506.245,1142,MA


In [99]:
# Extract GS AND MS data
tkr_1 = 'MA'
tkr_2 = 'V'
x_raw = rth[rth['symbol'] == tkr_1][['ts', 'close']].copy()
y_raw = rth[rth['symbol'] == tkr_2][['ts', 'close']].copy()

# Filter to only include data from June 1, 2025 onwards
start_date = pd.Timestamp('2025-04-01', tz='UTC')
end_date = pd.Timestamp('2025-05-01', tz='UTC')
x_raw = x_raw[(x_raw['ts'] >= start_date) & (x_raw['ts'] <= end_date)]
y_raw = y_raw[(y_raw['ts'] >= start_date) & (y_raw['ts'] <= end_date)]

# Convert to datetime index and resample to 5-minute intervals
# **(I'm trying experimenting with different timeframes)** to see if it changes cointegration results
x_raw['ts'] = pd.to_datetime(x_raw['ts'])
y_raw['ts'] = pd.to_datetime(y_raw['ts'])

x_raw = x_raw.set_index('ts')
y_raw = y_raw.set_index('ts')


In [100]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron 
import scipy.stats as stats
from scipy import signal
import warnings
warnings.filterwarnings('ignore')


In [101]:
def screen_pair(
    x, y,
    min_obs=100,
    bar_minutes=1,                 # 1 for 1-min bars; 5 for 5-min bars, etc.
    hl_min_minutes=10,
    hl_max_minutes=180
):
    """
    Engle–Granger style screening on a fixed window.
    x, y: aligned price series (prefer log-closes), same index, RTH only.
    """

    # --- 0) Basic checks & align ---
    if not isinstance(x, pd.Series): x = pd.Series(x)
    if not isinstance(y, pd.Series): y = pd.Series(y)

    idx = x.index.intersection(y.index)
    if len(idx) < min_obs:
        return dict(cointegrated=False, reason="Insufficient common observations", n_obs=len(idx))

    x = x.loc[idx].astype(float)
    y = y.loc[idx].astype(float)
    if x.isna().any() or y.isna().any():
        mask = (~x.isna()) & (~y.isna())
        x, y = x[mask], y[mask]
    n = len(x)
    if n < min_obs:
        return dict(cointegrated=False, reason="Insufficient data after NaN drop", n_obs=n)

    res = dict(n_obs=n)

    # --- 1) Static hedge: y = alpha + beta x + eps ---
    X = sm.add_constant(x.values)                   # intercept
    ols = sm.OLS(y.values, X).fit()
    alpha, beta = float(ols.params[0]), float(ols.params[1])
    u = y.values - (alpha + beta * x.values)        # residuals (spread candidate)
    res.update(hedge_ratio=beta, intercept=alpha, r_squared=float(ols.rsquared))

    # --- 2) Unit-root tests on residuals ---
    # ADF on residuals with NO constant/trend per EG
    adf = adfuller(u, regression='n', autolag='AIC')   # returns (stat, pval, usedlag, nobs, crit, icbest)
    adf_stat, adf_p, adf_crit = float(adf[0]), float(adf[1]), adf[4]
    adf_ok = (adf_p <= 0.05)

    # PP test (complements ADF; robust to serial corr/heteroskedasticity)
    pp_p = float(PhillipsPerron(u, trend='n').pvalue)  # 'n' = no constant
    pp_ok = (pp_p <= 0.05)

    # KPSS has "stationary" null; here we use it as a sanity check (don’t reject)
    try:
        kpss_stat, kpss_p, *_ = kpss(u, regression='c', nlags='auto')
        kpss_ok = (kpss_p >= 0.10)
    except Exception:
        kpss_stat, kpss_p, kpss_ok = np.nan, np.nan, False

    res.update(tests={
        'adf':   {'stat': adf_stat, 'pvalue': adf_p,  'ok': adf_ok, 'crit': adf_crit, 'reg': 'n'},
        'pp':    {'pvalue': pp_p,    'ok': pp_ok,     'reg': 'n'},
        'kpss':  {'stat': kpss_stat, 'pvalue': kpss_p,'ok': kpss_ok,'reg': 'c'},
    })

    # --- 3) Mean-reversion strength via AR(1) φ and half-life ---
    # φ = argmin ||u_{t+1} - φ u_t||^2  (no intercept)
    u0, u1 = u[:-1], u[1:]
    denom = (u0 @ u0)
    phi = float((u0 @ u1) / denom) if denom > 0 else np.nan

    # constrain for stability and compute half-life in *bars*
    if np.isfinite(phi) and 0 < phi < 1:
        hl_bars = np.log(2) / (-np.log(phi))
        hl_minutes = hl_bars * bar_minutes
        phi_ok = True
    else:
        hl_bars, hl_minutes, phi_ok = np.inf, np.inf, False

    res['tests']['ar1'] = {'phi': phi, 'phi_ok': phi_ok, 'half_life_bars': hl_bars, 'half_life_minutes': hl_minutes}

    # --- 4) Variance Ratio on residual differences (Δu) ---
    du = np.diff(u)
    def variance_ratio(d, k):
        if len(d) <= k: return np.nan
        # Simplified variance ratio calculation
        # Calculate k-period returns by summing consecutive differences
        n = len(d)
        k_returns = np.zeros(n - k + 1)
        for i in range(n - k + 1):
            k_returns[i] = np.sum(d[i:i+k])
        
        v1 = np.var(d, ddof=1)
        vk = np.var(k_returns, ddof=1) / k  # Normalize by k
        return float(vk / v1) if v1 > 0 else np.nan

    vr2, vr4, vr8 = map(lambda k: variance_ratio(du, k), (2,4,8))
    vr_ok = all([(v < 1) for v in (vr2, vr4, vr8) if np.isfinite(v)])

    res['tests']['variance_ratio'] = {'vr2': vr2, 'vr4': vr4, 'vr8': vr8, 'ok': vr_ok}

    # --- 5) Decision logic ---
    # Core requirement: EG/ADF must reject unit root; PP backs it up; KPSS shouldn’t reject stationarity.
    stationarity_pass = (adf_ok and pp_ok and kpss_ok)
    hl_ok = (hl_min_minutes <= hl_minutes <= hl_max_minutes)
    cointegrated = bool(stationarity_pass and phi_ok and hl_ok)

    res.update(
        stationarity_pass=stationarity_pass,
        mean_reversion_pass=(phi_ok and vr_ok),
        half_life_acceptable=hl_ok,
        cointegrated=cointegrated,
        reason=(
            "OK"
            if cointegrated else
            "; ".join([
                "" if stationarity_pass else "Residual not stationary (ADF/PP/KPSS)",
                "" if phi_ok else "AR(1) phi not in (0,1)",
                "" if hl_ok else f"Half-life {hl_minutes:.1f}m outside band [{hl_min_minutes},{hl_max_minutes}]",
                "" if vr_ok else "Variance ratio not < 1"
            ]).strip("; ").replace(";;",";")
        )
    )
    return res


In [102]:
# Test the cointegration screening function
# First, let's prepare the data properly for the function

# Extract close prices and set timestamp as index for proper alignment
# If 'ts' is already the index, just access 'close' directly
x_prices = x_raw['close']
y_prices = y_raw['close']

print(f"{tkr_1} data shape:", x_prices.shape)
print(f"{tkr_2} data shape:", y_prices.shape)
print(f"\nFirst few {tkr_1} prices:")
print(x_prices.head())
print(f"\nFirst few {tkr_2} prices:")
print(y_prices.head())


MA data shape: (7984,)
V data shape: (8179,)

First few MA prices:
ts
2025-04-01 13:30:00+00:00    546.630
2025-04-01 13:31:00+00:00    544.870
2025-04-01 13:32:00+00:00    546.800
2025-04-01 13:33:00+00:00    547.685
2025-04-01 13:34:00+00:00    547.180
Name: close, dtype: float64

First few V prices:
ts
2025-04-01 13:30:00+00:00    349.19
2025-04-01 13:31:00+00:00    348.38
2025-04-01 13:32:00+00:00    349.16
2025-04-01 13:33:00+00:00    348.82
2025-04-01 13:34:00+00:00    348.58
Name: close, dtype: float64


In [103]:
# Run the cointegration screening
results = screen_pair(x_prices, y_prices, bar_minutes=5)

print("=== COINTEGRATION SCREENING RESULTS ===")
print(f"Cointegrated: {results['cointegrated']}")
print(f"Reason: {results['reason']}")
print(f"Number of observations: {results['n_obs']}")

if 'hedge_ratio' in results:
    print(f"\nHedge Ratio (β): {results['hedge_ratio']:.4f}")
    print(f"Intercept (α): {results['intercept']:.4f}")
    print(f"R-squared: {results['r_squared']:.4f}")

print("\n=== DETAILED TEST RESULTS ===")

# Stationarity tests
if 'adf' in results['tests']:
    adf = results['tests']['adf']
    print(f"\nADF Test:")
    print(f"  Statistic: {adf['stat']:.4f}")
    print(f"  P-value: {adf['pvalue']:.4f}")
    print(f"  Stationary: {adf['ok']}")

if 'pp' in results['tests']:
    pp = results['tests']['pp']
    print(f"\nPhillips-Perron Test:")
    print(f"  P-value: {pp['pvalue']:.4f}")
    print(f"  Stationary: {pp['ok']}")

if 'kpss' in results['tests']:
    kpss = results['tests']['kpss']
    print(f"\nKPSS Test:")
    print(f"  Statistic: {kpss['stat']:.4f}")
    print(f"  P-value: {kpss['pvalue']:.4f}")
    print(f"  Stationary: {kpss['ok']}")

# Mean reversion tests
if 'ar1' in results['tests']:
    ar1 = results['tests']['ar1']
    print(f"\nAR(1) Model:")
    print(f"  Phi (AR coefficient): {ar1['phi']:.4f}")
    print(f"  Mean reverting: {ar1['phi_ok']}")
    print(f"  Half-life: {ar1['half_life_minutes']:.2f} minutes")

if 'variance_ratio' in results['tests']:
    vr = results['tests']['variance_ratio']
    print(f"\nVariance Ratio Test:")
    print(f"  VR(2): {vr['vr2']:.4f}")
    print(f"  VR(4): {vr['vr4']:.4f}")
    print(f"  VR(8): {vr['vr8']:.4f}")
    print(f"  Mean reverting: {vr['ok']}")

print(f"\n=== SUMMARY ===")
print(f"Stationarity passed: {results.get('stationarity_pass', 'N/A')}")
print(f"Mean reversion passed: {results.get('mean_reversion_pass', 'N/A')}")
print(f"Half-life acceptable: {results.get('half_life_acceptable', 'N/A')}")


=== COINTEGRATION SCREENING RESULTS ===
Cointegrated: False
Reason: Residual not stationary (ADF/PP/KPSS); ; Half-life 1634.9m outside band [10,180]
Number of observations: 7956

Hedge Ratio (β): 0.4938
Intercept (α): 75.1634
R-squared: 0.8777

=== DETAILED TEST RESULTS ===

ADF Test:
  Statistic: -1.9338
  P-value: 0.0507
  Stationary: False

Phillips-Perron Test:
  P-value: 0.0331
  Stationary: True

KPSS Test:
  Statistic: 2.6713
  P-value: 0.0100
  Stationary: False

AR(1) Model:
  Phi (AR coefficient): 0.9979
  Mean reverting: True
  Half-life: 1634.91 minutes

Variance Ratio Test:
  VR(2): 0.7687
  VR(4): 0.6217
  VR(8): 0.5514
  Mean reverting: True

=== SUMMARY ===
Stationarity passed: False
Mean reversion passed: True
Half-life acceptable: False
